# Train a LEGO minifigure detector for RoboControl

One-time training notebook — run this here in Colab (free GPU), then send the
downloaded `.tflite` + label file back so they can be wired into the robocar's
`app/hardware/ml_driver.py` (it already detects 80 COCO categories including
`person`; this adds a second, parallel single-class detector for
`lego_minifigure` that runs alongside it — the COCO model is untouched).

**Before running:** `Runtime` menu → `Change runtime type` → select a `GPU`.

Dataset: [Lego Minifigures](https://universe.roboflow.com/vc-echpj/lego-minifigures-r3zzt)
on Roboflow Universe (413 labeled images, CC BY 4.0) — built for detecting
*any* minifigure generically, not one specific figure.

In [ ]:
!pip install -q roboflow ultralytics

## Step 1 — download the dataset

1. Open the dataset page: https://universe.roboflow.com/vc-echpj/lego-minifigures-r3zzt
2. Click **Download Dataset** (top right)
3. Format: choose **YOLOv8**
4. Sign in / create a free Roboflow account if asked
5. It shows you a code snippet with *your own* API key — copy that whole
   snippet and paste it into the cell below, **replacing everything in it**.
   The version number in the snippet may not be `1` — use whatever it shows.

In [ ]:
# PASTE the snippet Roboflow's "Download Dataset" button gives you here,
# replacing this placeholder entirely. It will look like this, but with
# YOUR api_key and (possibly) a different version number:

from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_API_KEY_HERE")
project = rf.workspace("vc-echpj").project("lego-minifigures-r3zzt")
version = project.version(1)
dataset = version.download("yolov8")

print("Downloaded to:", dataset.location)

## Step 2 — train

`yolov8n` (nano) is the smallest/fastest base model — matters because this
runs on a Jetson Nano's CPU with no GPU acceleration. `imgsz=320` keeps
inference fast there too (the existing COCO model also runs at a similar
resolution). 60 epochs is a reasonable starting point for ~400 images —
re-run with more if the validation metrics printed at the end look weak.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60,
    imgsz=320,
    batch=16,
    name="lego_minifigure",
)

## Step 3 — export to TFLite (int8 quantized) and download

In [ ]:
exported_dir = model.export(format="tflite", int8=True, imgsz=320, data=f"{dataset.location}/data.yaml")
print("Exported:", exported_dir)

In [ ]:
import glob
from google.colab import files

candidates = glob.glob("runs/detect/lego_minifigure*/weights/best_saved_model/*_int8.tflite")
assert candidates, "No int8 .tflite found — check the export step's output above for the real path"
tflite_path = candidates[0]
print("Model file:", tflite_path)

with open("lego_labels.txt", "w") as f:
    f.write("lego_minifigure\n")

files.download(tflite_path)
files.download("lego_labels.txt")

## Step 4 — send the files back

Two files should now be downloaded to your computer:
- a `*_int8.tflite` model file
- `lego_labels.txt`

Drop both into the conversation (or tell me where you saved them) and the
integration into `app/hardware/ml_driver.py` happens on the robocar side —
nothing else needed here.